In [1]:
# ============================================================
# SRI LANKA TOURISM ANALYTICS
# NOTEBOOK 06 - SQL ANALYSIS + SQLITE
# ============================================================

import pandas as pd
import sqlite3
from pathlib import Path

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\DataAnalyticsPortfolio\01_Projects\01_Sri_Lanka_Tourism_Analytics")

INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "tourism_arrivals_clean.csv"

SQL_DIR = PROJECT_ROOT / "sql"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "sql"

SQL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_FILE = SQL_DIR / "tourism_analysis.db"

# ------------------------------------------------------------
# 2. LOAD CLEAN DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df["Country_Display"] = (
    df["Country_Standardized"]
    .fillna(df["Country_Raw"])
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 3. CREATE SQLITE DATABASE
# ------------------------------------------------------------

conn = sqlite3.connect(DB_FILE)

# Replace table if it already exists
df.to_sql(
    "tourism_arrivals",
    conn,
    if_exists="replace",
    index=False
)

# ------------------------------------------------------------
# 4. CREATE INDEXES
# ------------------------------------------------------------

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_tourism_country
ON tourism_arrivals(Country_Display)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_tourism_year
ON tourism_arrivals(Year)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_tourism_date
ON tourism_arrivals(Date)
""")

conn.commit()

# ------------------------------------------------------------
# 5. HELPER FUNCTION
# ------------------------------------------------------------

def run_sql(query, output_file=None):
    result = pd.read_sql_query(query, conn)

    if output_file:
        result.to_csv(OUTPUT_DIR / output_file, index=False)

    return result


# ============================================================
# SQL ANALYSIS
# ============================================================

print("=" * 70)
print("SRI LANKA TOURISM - SQL ANALYSIS + SQLITE")
print("=" * 70)

print(f"\nDatabase created:")
print(DB_FILE)

print(f"\nRows loaded into SQLite: {len(df):,}")
print(f"Countries: {df['Country_Display'].nunique():,}")


# ------------------------------------------------------------
# QUERY 1 - DATASET SUMMARY
# ------------------------------------------------------------

q1 = """
SELECT
    COUNT(*) AS Total_Rows,
    COUNT(DISTINCT Country_Display) AS Countries,
    MIN(Year) AS First_Year,
    MAX(Year) AS Last_Year,
    MIN(Date) AS First_Date,
    MAX(Date) AS Last_Date,
    SUM(CASE WHEN Arrivals IS NULL THEN 1 ELSE 0 END) AS Missing_Arrivals
FROM tourism_arrivals;
"""

summary = run_sql(q1, "01_dataset_summary.csv")

print("\n1. DATASET SUMMARY")
print(summary.to_string(index=False))


# ------------------------------------------------------------
# QUERY 2 - YEARLY ARRIVALS
# ------------------------------------------------------------

q2 = """
SELECT
    Year,
    SUM(Arrivals) AS Total_Arrivals
FROM tourism_arrivals
GROUP BY Year
ORDER BY Year;
"""

yearly = run_sql(q2, "02_yearly_arrivals.csv")

print("\n2. YEARLY TOURIST ARRIVALS")
print(yearly.to_string(index=False))


# ------------------------------------------------------------
# QUERY 3 - MONTHLY ARRIVALS
# ------------------------------------------------------------

q3 = """
SELECT
    Year,
    Month_Number,
    Month,
    SUM(Arrivals) AS Total_Arrivals
FROM tourism_arrivals
GROUP BY Year, Month_Number, Month
ORDER BY Year, Month_Number;
"""

monthly = run_sql(q3, "03_monthly_arrivals.csv")

print("\n3. MONTHLY TOURIST ARRIVALS")
print(monthly.head(20).to_string(index=False))


# ------------------------------------------------------------
# QUERY 4 - TOP SOURCE MARKETS
# ------------------------------------------------------------

q4 = """
SELECT
    Country_Display AS Country,
    SUM(Arrivals) AS Total_Arrivals,
    ROUND(
        100.0 * SUM(Arrivals) /
        (SELECT SUM(Arrivals)
         FROM tourism_arrivals),
        2
    ) AS Share_Percent
FROM tourism_arrivals
GROUP BY Country_Display
ORDER BY Total_Arrivals DESC
LIMIT 15;
"""

top_countries = run_sql(q4, "04_top_source_markets.csv")

print("\n4. TOP 15 SOURCE MARKETS")
print(top_countries.to_string(index=False))


# ------------------------------------------------------------
# QUERY 5 - COUNTRY RANKING BY YEAR
# ------------------------------------------------------------

q5 = """
WITH country_year AS (
    SELECT
        Year,
        Country_Display AS Country,
        SUM(Arrivals) AS Arrivals
    FROM tourism_arrivals
    GROUP BY Year, Country_Display
),

ranked AS (
    SELECT
        Year,
        Country,
        Arrivals,
        RANK() OVER (
            PARTITION BY Year
            ORDER BY Arrivals DESC
        ) AS Country_Rank
    FROM country_year
)

SELECT *
FROM ranked
WHERE Country_Rank <= 10
ORDER BY Year, Country_Rank;
"""

country_ranking = run_sql(q5, "05_country_ranking_by_year.csv")

print("\n5. TOP 10 SOURCE MARKETS BY YEAR")
print(country_ranking.to_string(index=False))


# ------------------------------------------------------------
# QUERY 6 - 2026 JAN-OCT TOP MARKETS
# ------------------------------------------------------------

q6 = """
SELECT
    Country_Display AS Country,
    SUM(Arrivals) AS Arrivals,
    ROUND(
        100.0 * SUM(Arrivals) /
        (
            SELECT SUM(Arrivals)
            FROM tourism_arrivals
            WHERE Year = 2026
              AND Month_Number <= 10
        ),
        2
    ) AS Share_Percent
FROM tourism_arrivals
WHERE Year = 2026
  AND Month_Number <= 10
GROUP BY Country_Display
ORDER BY Arrivals DESC
LIMIT 15;
"""

top_2026 = run_sql(q6, "06_top_markets_2026_jan_oct.csv")

print("\n6. TOP 15 SOURCE MARKETS - 2026 JAN-OCT")
print(top_2026.to_string(index=False))


# ------------------------------------------------------------
# QUERY 7 - JAN-OCT YEAR COMPARISON
# ------------------------------------------------------------

q7 = """
SELECT
    Year,
    SUM(Arrivals) AS Jan_Oct_Arrivals
FROM tourism_arrivals
WHERE Month_Number <= 10
GROUP BY Year
ORDER BY Year;
"""

jan_oct = run_sql(q7, "07_jan_oct_comparison.csv")

print("\n7. JANUARY-OCTOBER COMPARISON")
print(jan_oct.to_string(index=False))


# ------------------------------------------------------------
# QUERY 8 - SEASONAL MONTH ANALYSIS
# ------------------------------------------------------------

q8 = """
SELECT
    Month_Number,
    Month,
    ROUND(AVG(Arrivals), 2) AS Average_Monthly_Arrivals,
    MIN(Arrivals) AS Minimum_Arrivals,
    MAX(Arrivals) AS Maximum_Arrivals
FROM tourism_arrivals
GROUP BY Month_Number, Month
ORDER BY Month_Number;
"""

seasonality = run_sql(q8, "08_seasonality.csv")

print("\n8. SEASONALITY ANALYSIS")
print(seasonality.to_string(index=False))


# ------------------------------------------------------------
# QUERY 9 - 2022 TO 2025 MARKET GROWTH
# ------------------------------------------------------------

q9 = """
WITH country_year AS (
    SELECT
        Country_Display AS Country,
        Year,
        SUM(Arrivals) AS Arrivals
    FROM tourism_arrivals
    WHERE Year IN (2022, 2025)
    GROUP BY Country_Display, Year
),

pivoted AS (
    SELECT
        Country,
        MAX(CASE WHEN Year = 2022 THEN Arrivals END) AS Arrivals_2022,
        MAX(CASE WHEN Year = 2025 THEN Arrivals END) AS Arrivals_2025
    FROM country_year
    GROUP BY Country
)

SELECT
    Country,
    Arrivals_2022,
    Arrivals_2025,
    ROUND(
        100.0 * (Arrivals_2025 - Arrivals_2022)
        / NULLIF(Arrivals_2022, 0),
        2
    ) AS Growth_Percent,
    (Arrivals_2025 - Arrivals_2022) AS Absolute_Change
FROM pivoted
WHERE Arrivals_2022 >= 10000
ORDER BY Growth_Percent DESC;
"""

growth = run_sql(q9, "09_country_growth_2022_2025.csv")

print("\n9. FASTEST-GROWING MARKETS")
print(growth.head(15).to_string(index=False))


# ------------------------------------------------------------
# QUERY 10 - MONTHLY PERFORMANCE BY YEAR
# ------------------------------------------------------------

q10 = """
SELECT
    Month_Number,
    Month,
    SUM(CASE WHEN Year = 2022 THEN Arrivals ELSE 0 END) AS Arrivals_2022,
    SUM(CASE WHEN Year = 2023 THEN Arrivals ELSE 0 END) AS Arrivals_2023,
    SUM(CASE WHEN Year = 2024 THEN Arrivals ELSE 0 END) AS Arrivals_2024,
    SUM(CASE WHEN Year = 2025 THEN Arrivals ELSE 0 END) AS Arrivals_2025,
    SUM(CASE WHEN Year = 2026 THEN Arrivals ELSE 0 END) AS Arrivals_2026
FROM tourism_arrivals
GROUP BY Month_Number, Month
ORDER BY Month_Number;
"""

monthly_year = run_sql(q10, "10_monthly_performance_by_year.csv")

print("\n10. MONTHLY PERFORMANCE BY YEAR")
print(monthly_year.to_string(index=False))


# ------------------------------------------------------------
# QUERY 11 - TOP 10 MARKETS IN 2025
# ------------------------------------------------------------

q11 = """
SELECT
    Country_Display AS Country,
    SUM(Arrivals) AS Arrivals
FROM tourism_arrivals
WHERE Year = 2025
GROUP BY Country_Display
ORDER BY Arrivals DESC
LIMIT 10;
"""

top_2025 = run_sql(q11, "11_top_markets_2025.csv")

print("\n11. TOP 10 MARKETS - 2025")
print(top_2025.to_string(index=False))


# ------------------------------------------------------------
# QUERY 12 - TOP 10 MARKETS IN 2022
# ------------------------------------------------------------

q12 = """
SELECT
    Country_Display AS Country,
    SUM(Arrivals) AS Arrivals
FROM tourism_arrivals
WHERE Year = 2022
GROUP BY Country_Display
ORDER BY Arrivals DESC
LIMIT 10;
"""

top_2022 = run_sql(q12, "12_top_markets_2022.csv")

print("\n12. TOP 10 MARKETS - 2022")
print(top_2022.to_string(index=False))


# ------------------------------------------------------------
# DATABASE VALIDATION
# ------------------------------------------------------------

table_check = pd.read_sql_query("""
SELECT
    name
FROM sqlite_master
WHERE type = 'table';
""", conn)

print("\nDATABASE TABLES")
print(table_check.to_string(index=False))


# ------------------------------------------------------------
# SAVE SQL QUERIES FOR PORTFOLIO
# ------------------------------------------------------------

queries = {
    "01_dataset_summary.sql": q1,
    "02_yearly_arrivals.sql": q2,
    "03_monthly_arrivals.sql": q3,
    "04_top_source_markets.sql": q4,
    "05_country_ranking_by_year.sql": q5,
    "06_top_markets_2026_jan_oct.sql": q6,
    "07_jan_oct_comparison.sql": q7,
    "08_seasonality.sql": q8,
    "09_country_growth_2022_2025.sql": q9,
    "10_monthly_performance_by_year.sql": q10,
    "11_top_markets_2025.sql": q11,
    "12_top_markets_2022.sql": q12
}

for filename, query in queries.items():
    (SQL_DIR / filename).write_text(query.strip() + "\n", encoding="utf-8")


# ------------------------------------------------------------
# CLOSE DATABASE
# ------------------------------------------------------------

conn.close()

print("\n" + "=" * 70)
print("SQL ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"\nSQLite database:")
print(DB_FILE)

print(f"\nSQL query files created: {len(queries)}")
print(f"CSV analysis files created: {len(list(OUTPUT_DIR.glob('*.csv')))}")

print("\nImportant:")
print("2026 = January-October only.")
print("2022-2025 = complete-year data.")
print("\nNext stage: Visualization and reporting")

SRI LANKA TOURISM - SQL ANALYSIS + SQLITE

Database created:
C:\DataAnalyticsPortfolio\01_Projects\01_Sri_Lanka_Tourism_Analytics\sql\tourism_analysis.db

Rows loaded into SQLite: 11,110
Countries: 229

1. DATASET SUMMARY
 Total_Rows  Countries  First_Year  Last_Year          First_Date           Last_Date  Missing_Arrivals
      11110        229        2022       2026 2022-01-01 00:00:00 2026-10-01 00:00:00                 2

2. YEARLY TOURIST ARRIVALS
 Year  Total_Arrivals
 2022        719978.0
 2023       1487303.0
 2024       2053465.0
 2025       2362521.0
 2026       1862304.0

3. MONTHLY TOURIST ARRIVALS
 Year  Month_Number     Month  Total_Arrivals
 2022             1   January         82327.0
 2022             2  February         96507.0
 2022             3     March        106500.0
 2022             4     April         62980.0
 2022             5       May         30207.0
 2022             6      June         32856.0
 2022             7      July         47293.0
 2022        